# Generative Text Model using GPT-2
### Internship Project — Natural Language Processing

---

**Objective:** Build a text generation model that produces coherent, meaningful paragraphs based on user-provided topics.

**Approach:** We use **GPT-2** (Generative Pre-trained Transformer 2) via Hugging Face Transformers — a state-of-the-art language model trained on 40GB of internet text.

**How to Run:** Execute each cell in order. No GPU required — runs on CPU in Google Colab.

---

## 1. Background: GPT vs LSTM

### Why GPT-2 over LSTM?

Two main architectures exist for text generation:

| Feature | GPT-2 (Chosen) | LSTM |
|---------|----------------|------|
| Training required | No (pretrained) | Yes (hours/days) |
| Output quality | Excellent | Depends on data |
| Setup complexity | Low | High |
| GPU required | No (CPU works) | Strongly recommended |
| Understands long context | Yes (attention) | Limited (vanishing gradient) |

**LSTM (Long Short-Term Memory)** processes text sequentially, maintaining a 'memory cell'. While it handles sequences well, it struggles with long-range dependencies and requires significant training data and compute time.

**GPT-2 (Transformer-based)** uses self-attention to relate every word to every other word simultaneously. It was pretrained on 40GB of internet text, so we get production-quality outputs without any training.

**Decision:** GPT-2 via Hugging Face is used for this project due to superior output quality, zero training requirement, and simplicity of implementation.

## 2. Setup & Installation

In [ ]:
# Install required libraries
# transformers: Hugging Face library with GPT-2 and 10,000+ other models
# accelerate: Speeds up model loading and inference
!pip install transformers accelerate sentencepiece -q

## 3. Import Libraries & Load Model

In [ ]:
# Core imports
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

# ---------------------------------------------------------
# Model Options (comment/uncomment to switch):
#   'gpt2'        → 117M parameters (fast, good for Colab)
#   'gpt2-medium' → 345M parameters (better quality)
#   'gpt2-large'  → 774M parameters (best, needs more RAM)
# ---------------------------------------------------------
MODEL_NAME = "gpt2"

print(f"\nLoading {MODEL_NAME}...")

# Load tokenizer — converts text ↔ token IDs
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# Load the language model itself
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# Set to evaluation mode (disables training-specific operations like dropout)
model.eval()

# Use GPU if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✓ Model loaded successfully!")
print(f"✓ Running on: {device.upper()}")
print(f"✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Data Preprocessing — Understanding Tokenization

Before text enters the model, it must be converted to **tokens** — numerical IDs the model understands.

GPT-2 uses **Byte-Pair Encoding (BPE)**: words are split into common subword units.

Example:
- `"artificial"` → `["art", "ific", "ial"]` → `[আর্ট, ific, ial]` (token IDs)
- Common words like `"the"`, `"is"` → single token
- Rare words → multiple tokens

This allows the model to handle any word, even made-up ones!

In [ ]:
# ---- Tokenization Demo ----

sample_text = "Artificial intelligence is transforming our world rapidly."

# Step 1: Encode text to token IDs
token_ids = tokenizer.encode(sample_text)
print("Original text:")
print(f"  '{sample_text}'")
print(f"\nToken IDs ({len(token_ids)} tokens):")
print(f"  {token_ids}")

# Step 2: See each token as text
print("\nToken breakdown:")
print(f"  {'Token ID':<12} {'Token Text':<20}")
print(f"  {'-'*12} {'-'*20}")
for tid in token_ids:
    token_text = tokenizer.decode([tid])
    print(f"  {tid:<12} '{token_text}'")

# Step 3: Decode back
decoded = tokenizer.decode(token_ids)
print(f"\nDecoded back: '{decoded}'")
print(f"\nVocabulary size: {tokenizer.vocab_size:,} unique tokens")

## 5. Model Architecture Overview

In [ ]:
# Print model configuration — all architecture details
config = model.config

print("=" * 50)
print(f"  GPT-2 Architecture Summary")
print("=" * 50)
print(f"  Model type         : {config.model_type}")
print(f"  Vocabulary size    : {config.vocab_size:,} tokens")
print(f"  Context length     : {config.n_positions} tokens max")
print(f"  Embedding size     : {config.n_embd} dimensions")
print(f"  Transformer layers : {config.n_layer}")
print(f"  Attention heads    : {config.n_head} per layer")
print(f"  Total parameters   : {sum(p.numel() for p in model.parameters()):,}")
print("=" * 50)

print("""
How it works:
  1. Input tokens → Embedding layer (maps IDs to 768-dim vectors)
  2. 12 Transformer blocks, each with:
     • Multi-head Self-Attention (12 heads look at different aspects)
     • Feed-Forward Network (learns complex patterns)
     • Layer Normalization (stabilizes training)
  3. Output layer → Probability distribution over 50,257 tokens
  4. Sample the next token → repeat until done
""")

## 6. Text Generation Function

The core function that generates text. Key parameters:

- **`temperature`**: Controls randomness. Low (0.5) = focused/repetitive. High (1.2) = creative/unpredictable. **0.8 is the sweet spot.**
- **`top_k`**: Sample only from the top-k most likely tokens. Prevents very unlikely words.
- **`top_p`** (nucleus sampling): Sample from the smallest set of tokens whose cumulative probability exceeds p. More dynamic than top_k.
- **`no_repeat_ngram_size`**: Prevents the model from repeating the same phrase.

In [ ]:
def generate_paragraph(prompt, max_length=200, temperature=0.8,
                       top_k=50, top_p=0.92, num_sequences=1):
    """
    Generate coherent paragraphs from a topic prompt using GPT-2.
    
    Args:
        prompt        (str)  : Starting text / topic
        max_length    (int)  : Total token count (prompt + generated)
        temperature   (float): Creativity — lower=focused, higher=creative (0.1-1.5)
        top_k         (int)  : Sample from top-k most likely tokens
        top_p         (float): Nucleus sampling cumulative probability cutoff
        num_sequences (int)  : Number of different outputs to generate
    
    Returns:
        list: Generated text strings
    """
    # Step 1: Convert prompt text to token IDs
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    inputs = inputs.to(device)  # Move to GPU/CPU
    
    # Step 2: Set padding token (required by GPT-2)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Step 3: Generate — no_grad means we don't compute gradients
    # (We're not training, just doing inference)
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,           # Stop after this many tokens
            temperature=temperature,         # Sampling temperature
            top_k=top_k,                     # Only consider top-k tokens
            top_p=top_p,                     # Nucleus sampling
            do_sample=True,                  # Sample (not greedy/deterministic)
            num_return_sequences=num_sequences,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,          # Prevent 3-gram repetition
            repetition_penalty=1.1           # Penalize repeated tokens
        )
    
    # Step 4: Decode token IDs back to readable text
    results = []
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        results.append(text)
    
    return results


# Quick test
print("Testing the generation function...")
test_output = generate_paragraph("The future of technology")
print("\n" + "="*60)
print(test_output[0])
print("="*60)
print("\n✓ Generation function working!")

## 7. Topic-Based Generator with Style Control

In [ ]:
def generate_on_topic(topic, style="paragraph", temperature=0.8):
    """
    Generate text on a user-specified topic in a chosen style.
    
    Args:
        topic       (str): The subject to write about
        style       (str): 'paragraph', 'essay', 'story', 'news', or 'qa'
        temperature (float): Creativity level (0.5 = focused, 1.0 = creative)
    
    Returns:
        str: Generated text
    """
    # Different prompt templates for different styles
    # A good prompt helps the model understand what format to use
    prompt_templates = {
        "paragraph": f"{topic} is important because",
        "essay":     f"Essay on {topic}:\n\nIntroduction:",
        "story":     f"Once upon a time, {topic.lower()}",
        "news":      f"Breaking News: {topic} —",
        "qa":        f"Question: What is {topic}?\nAnswer:"
    }
    
    prompt = prompt_templates.get(style, prompt_templates["paragraph"])
    
    print(f"📝 Topic      : {topic}")
    print(f"📄 Style      : {style}")
    print(f"🌡️  Temperature: {temperature}")
    print(f"🔤 Prompt     : {prompt!r}")
    print("\n" + "-" * 70)
    
    results = generate_paragraph(
        prompt,
        max_length=250,
        temperature=temperature,
        num_sequences=1
    )
    
    print(results[0])
    print("-" * 70 + "\n")
    
    return results[0]


print("✓ generate_on_topic() function defined")

## 8. Results — Sample Outputs on Different Topics

Run the model on diverse topics to demonstrate its generalization ability.

In [ ]:
# ---- Sample 1: Climate Change (Paragraph Style) ----
print("=" * 70)
print("SAMPLE 1")
print("=" * 70)
output1 = generate_on_topic("Climate change", style="paragraph", temperature=0.8)

In [ ]:
# ---- Sample 2: Machine Learning (Essay Style) ----
print("=" * 70)
print("SAMPLE 2")
print("=" * 70)
output2 = generate_on_topic("Machine learning", style="essay", temperature=0.75)

In [ ]:
# ---- Sample 3: Robots (Story Style) ----
print("=" * 70)
print("SAMPLE 3")
print("=" * 70)
output3 = generate_on_topic("A robot discovering art", style="story", temperature=1.0)

In [ ]:
# ---- Sample 4: Space Exploration (News Style) ----
print("=" * 70)
print("SAMPLE 4")
print("=" * 70)
output4 = generate_on_topic("Space exploration", style="news", temperature=0.7)

## 9. Temperature Comparison Experiment

Below we show how **temperature** changes the output style on the same prompt.
This demonstrates the model's controllability — a key feature of the system.

In [ ]:
# Temperature comparison experiment
prompt = "The ocean is"
temperatures = [0.5, 0.8, 1.1]
descriptions = ["Focused/conservative", "Balanced (recommended)", "Creative/unpredictable"]

print(f"Prompt: '{prompt}'")
print("=" * 70)

for temp, desc in zip(temperatures, descriptions):
    results = generate_paragraph(prompt, max_length=120, temperature=temp)
    print(f"\n🌡️  Temperature = {temp} ({desc})")
    print("-" * 40)
    print(results[0])

print("\n" + "=" * 70)
print("Observation: Higher temperature = more creative but less predictable.")

## 10. Interactive Generator (Try Your Own Topic!)

Change the `topic` and `style` variables below to generate text on any subject.

In [ ]:
# ---- CHANGE THESE VALUES ----
YOUR_TOPIC = "Quantum computing"    # <-- change this!
YOUR_STYLE = "paragraph"             # options: paragraph, essay, story, news, qa
YOUR_TEMPERATURE = 0.8              # 0.5 (focused) to 1.2 (creative)
# -----------------------------

your_output = generate_on_topic(YOUR_TOPIC, style=YOUR_STYLE, temperature=YOUR_TEMPERATURE)

## 11. Bonus: Gradio Web Interface

Deploy your model as an interactive web app with just a few lines of code!

In [ ]:
# Optional: Build a simple web UI for the model
# Uncomment and run if you want an interactive demo

# !pip install gradio -q
# import gradio as gr

# def gradio_generate(topic, style, temperature):
#     """Wrapper for the Gradio interface."""
#     results = generate_paragraph(
#         prompt=f"{topic} is",
#         max_length=250,
#         temperature=temperature
#     )
#     return results[0]

# demo = gr.Interface(
#     fn=gradio_generate,
#     inputs=[
#         gr.Textbox(label="Topic", placeholder="e.g. Climate change"),
#         gr.Dropdown(["paragraph", "essay", "story", "news"], label="Style"),
#         gr.Slider(0.3, 1.5, value=0.8, label="Temperature")
#     ],
#     outputs=gr.Textbox(label="Generated Text", lines=8),
#     title="GPT-2 Text Generator",
#     description="Generate coherent text on any topic using GPT-2!"
# )
# demo.launch()

print("Gradio demo code ready — uncomment to run!")

## 12. Analysis & Discussion

### What Works Well
- GPT-2 produces grammatically correct, coherent sentences on most topics
- The model generalizes across domains (science, storytelling, journalism)
- Temperature control gives us fine-grained creativity control
- No training required — ready to use immediately

### Limitations
- **Factual accuracy**: GPT-2 may generate plausible-sounding but incorrect facts
- **Context length**: Limited to 1024 tokens — struggles with very long documents
- **Topic specificity**: Without fine-tuning, outputs can drift from the intended topic
- **Bias**: Trained on internet data, may reflect societal biases

### Ethical Considerations
- Generated text can be used to spread misinformation — content filtering is important
- Models trained on biased data may produce biased outputs
- Attribution matters — AI-generated content should be labeled as such

## 13. Conclusion & Future Work

This project successfully demonstrates text generation using the GPT-2 transformer model. The system generates coherent, grammatically sound paragraphs on arbitrary topics without any training.

### Potential Improvements
1. **Fine-tuning** on domain-specific data (medical, legal, scientific) for better accuracy
2. **Larger model** (GPT-2 Medium/Large or GPT-Neo 1.3B) for better coherence
3. **Prompt engineering** with system context for better topic adherence
4. **RLHF** (Reinforcement Learning from Human Feedback) to reduce hallucinations
5. **Evaluation metrics**: BLEU score, perplexity, BERTScore for objective quality measurement
6. **Output filtering**: Add content moderation before displaying to users
7. **Streaming output**: Generate and display text token-by-token for better UX

### References
- Radford et al. (2019). Language Models are Unsupervised Multitask Learners. OpenAI Blog.
- Vaswani et al. (2017). Attention Is All You Need. NeurIPS.
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers